In [1]:
import numpy as np
import pandas as pd
from haversine import haversine

# 1. 청룡동

In [2]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '청룡동']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('청룡동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('청룡동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('청룡동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('청룡동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [3]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(138, 2)


array([[ 36.80013535, 127.1776807 ],
       [ 36.76749349, 127.1650992 ]])

In [4]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [5]:
def pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points):
    
    ### 버스
    havers1 = []
    for i in 후보지_points:
        site1 = []
        for j in 버스_points:
            site1.append(haversine(i,j)*1000)
        havers1.append(site1)

    location = list(입지후보지['이름'])
    location1 = list(버스['정류장명'])

    havers_D1 = dict(zip(location, [dict(zip(location1, i)) for i in havers1]))
    D1 = pd.DataFrame(havers_D1)
    
    최소값 = list(D1.min(axis=1))

    # 최단거리인 경우 가중치 부여, 아니면 0
    idx = D1.index
    col = D1.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D1.loc[idx[i], col[j]] == 최소값[i]:
                D1.loc[idx[i], col[j]] = m1
            else:
                D1.loc[idx[i], col[j]] = 0
    
    
     ### 노인시설
    havers2 = []
    for i in 후보지_points:
        site2 = []
        for j in 노인시설_points:
            site2.append(haversine(i,j)*1000)
        havers2.append(site2)

    location2 = list(노인시설['시설명'])

    havers_D2 = dict(zip(location, [dict(zip(location2, i)) for i in havers2]))
    D2 = pd.DataFrame(havers_D2)
    
    최소값 = list(D2.min(axis=1))

    idx = D2.index
    col = D2.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D2.loc[idx[i], col[j]] == 최소값[i]:
                D2.loc[idx[i], col[j]] = m2
            else:
                D2.loc[idx[i], col[j]] = 0


    ### 노인복지시설설
    havers3 = []
    for i in 후보지_points:
        site3 = []
        for j in 노인복지시설_points:
            site3.append(haversine(i,j)*1000)
        havers3.append(site3)

    location3 = list(노인복지시설['이름'])

    havers_D3 = dict(zip(location, [dict(zip(location3, i)) for i in havers3]))
    D3 = pd.DataFrame(havers_D3)
    
    최소값 = list(D3.min(axis=1))

    idx = D3.index
    col = D3.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D3.loc[idx[i], col[j]] == 최소값[i]:
                D3.loc[idx[i], col[j]] = m3
            else:
                D3.loc[idx[i], col[j]] = 0
             
                 
    ### 경로당
    havers4 = []
    for i in 후보지_points:
        site4 = []
        for j in 경로당_points:
            site4.append(haversine(i,j)*1000)
        havers4.append(site4)

    location4 = list(경로당['경로당명'])

    havers_D4 = dict(zip(location, [dict(zip(location4, i)) for i in havers4]))
    D4 = pd.DataFrame(havers_D4)
    
    최소값 = list(D4.min(axis=1))

    idx = D4.index
    col = D4.columns
    lidx = len(idx)
    lcol = len(col)

    for i in range(lidx):
        for j in range(lcol):
            if D4.loc[idx[i], col[j]] == 최소값[i]:
                D4.loc[idx[i], col[j]] = m4
            else:
                D4.loc[idx[i], col[j]] = 0
                
    D_final = pd.concat([D1, D2, D3, D4])
    print(D_final.sum().sort_values(ascending=False))

    return D_final

In [6]:
results = pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

results = results.sum().sort_values(ascending=False)

최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())][['이름', '위도', '경도', '주소']].copy()

최종후보지.to_csv('청룡동_후보지(P-median).csv', index=False, encoding='utf-8-sig')

청당코오롱하늘채                 5.644928
구성1공원                    5.398551
하천변                      5.144928
천안삼거리공원 (2025년 9월 예정)    4.818841
청수2공원                    3.318841
청당1공원                    3.152174
청룡동 주민센터                 2.985507
구룡3통 마을회관 앞 정자           2.572464
청당3공원                    2.572464
뒷들공원(청당7공원)              2.239130
청수호수공원                   1.992754
천안생활체육공원                 1.913043
구성2공원                    1.826087
청당골공원(청당5공원)             1.659420
마을회관                     1.492754
한양수자인블루시티(아)             0.913043
청수1공원                    0.746377
청수제2공원                   0.746377
청당2공원                    0.746377
청수산림공원                   0.746377
청당6공원                    0.746377
dtype: float64


# 2. 성환읍

In [7]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '성환읍']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('성환읍_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('성환읍_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('성환읍_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('성환읍_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [8]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(256, 2)


array([[ 36.94717367, 127.1513989 ],
       [ 36.92117221, 127.1374003 ]])

In [9]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [10]:
results = pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

results = results.sum().sort_values(ascending=False)

최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())][['이름', '위도', '경도', '주소']].copy()

최종후보지.to_csv('성환읍_후보지(P-median).csv', index=False, encoding='utf-8-sig')

수향1리경로회관              16.339844
아파트단지                  7.187500
성환읍사무소                 7.011719
노인회관                   6.175781
신가2리 복지회관              5.343750
창환주택 단지 안              4.964844
왕림3리 노인회관              4.332031
마을회관                   4.054688
노인회관 옆                 3.132812
금곡쉼터(율금3리)             3.085938
노인건강쉼터(성환3리)           3.042969
북부스포츠센터 인라인스케이트장 옆     2.800781
양령2리마을회관               2.488281
화친정(양령3리)              2.398438
성환4공원                  2.210938
도로변(연암율금로)             1.566406
대한노인회성환읍분회             1.476562
매주10리(부영3차아파트)         1.199219
매주7리(부영1차아파트)          0.921875
공원                     0.644531
등산로                    0.644531
마을공터                   0.644531
보건진료소                  0.644531
매주9리(부영2차아파트)          0.000000
dtype: float64


# 3. 신안동

In [11]:
입지후보지 = pd.read_csv('입지후보지.csv')
입지후보지 = 입지후보지[입지후보지['행정동'] == '신안동']
후보지_points = np.array([list(i) for i in zip(입지후보지['위도'], 입지후보지['경도'])])

버스 = pd.read_csv('신안동_버스정류장.csv')
버스_points = np.array([list(i) for i in zip(버스['Y좌표'], 버스['X좌표'])])

노인시설 = pd.read_csv('신안동_노인시설.csv', index_col=0)
노인시설_points = np.array([list(i) for i in zip(노인시설['위도'], 노인시설['경도'])])

노인복지시설 = pd.read_csv('신안동_노인복지시설.csv', index_col=0)
노인복지시설_points = np.array([list(i) for i in zip(노인복지시설['위도'], 노인복지시설['경도'])])

경로당 = pd.read_csv('신안동_경로당.csv', index_col=0)
경로당_points = np.array([list(i) for i in zip(경로당['위도'], 경로당['경도'])])
경로당 = 경로당.reset_index()

In [12]:
X = list(노인시설['위도']) + list(노인복지시설['위도']) + list(경로당['위도']) + list(버스['Y좌표'])
Y = list(노인시설['경도']) + list(노인복지시설['경도']) + list(경로당['경도']) + list(버스['X좌표'])
points = np.array([list(i) for i in zip(X, Y)])
print(points.shape)
points[:2]

(98, 2)


array([[ 36.81772127, 127.1525376 ],
       [ 36.81753212, 127.1518383 ]])

In [13]:
전체w = points.shape[0]
버스w = 버스.shape[0]
노인시설w = 노인시설.shape[0]
노인복지시설w = 노인복지시설.shape[0]
경로당w = 경로당.shape[0]

# 가중치는 해당수요지점수 / 전체수요지점수

m1 = 버스w / 전체w
m2 = 노인시설w / 전체w
m3 = 노인복지시설w / 전체w
m4 = 경로당w / 전체w

In [14]:
results = pmedian(후보지_points, 버스_points, 노인시설_points, 노인복지시설_points, 경로당_points)

results = results.sum().sort_values(ascending=False)

최종후보지 = 입지후보지[입지후보지['이름'].isin(results.head(5).index.tolist())][['이름', '위도', '경도', '주소']].copy()

최종후보지.to_csv('신안동_후보지(P-median).csv', index=False, encoding='utf-8-sig')

안서1공원        7.071429
신부6공원        4.459184
안서3어린이공원     4.275510
신부문화공원       3.346939
천호지근린공원      2.418367
노인회관         2.234694
마을공터         2.234694
신부공원         1.489796
성정9공원        1.489796
안서2어린이공원     1.489796
신부2공원        1.112245
도솔공원         0.744898
신안동주민센터      0.367347
편지공원         0.367347
신부8공원        0.183673
마을회관         0.010204
태조산공원        0.000000
도솔광장 족구장     0.000000
천호지생활체육공원    0.000000
천안향교 앞       0.000000
신부1공원        0.000000
도솔광장         0.000000
dtype: float64
